In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
for metin in ["gelcekmsn", "halısaya aldık", "b3dava", "Değerli müşterimiz kartınız bloke oldu [LINK] tıklayın"]:
    print(tok.tokenize(metin))
    

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
model = model.to("cuda")

print(sum(p.numel() for p in model.parameters()) / 1e6, "milyon parametre")
print(torch.cuda.memory_allocated() / 1e9, "GB GPU belleği kullanılıyor")

In [ ]:
import sys
sys.path.append("..")
import pandas as pd

train = pd.read_csv("../data/processed/tr_train.csv")
train = train[train["belirsiz"] != "e"]

uzunluklar = train["mesaj"].apply(lambda m: len(tok(m)["input_ids"]))
print(uzunluklar.describe())
print("95. yüzdelik:", uzunluklar.quantile(0.95))




In [ ]:
import time
import numpy as np
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

del model
torch.cuda.empty_cache()

AYAR = {"model": "FacebookAI/xlm-roberta-base", "max_len": 80, "batch": 16,
        "lr": 2e-5, "epoch": 4, "warmup": 0.1, "weight_decay": 0.01, "seed": 42}

X = train["mesaj"].tolist()
y = (train["ikili"] == "spam").astype(int).to_numpy()
katlar = train["fold"].to_numpy()

def yukleyici(metinler, etiketler, a, karistir):
    veri = list(zip(metinler, etiketler))
    def birlestir(parti):
        m, e = zip(*parti)
        kod = tok(list(m), padding=True, truncation=True, max_length=a["max_len"], return_tensors="pt")
        kod["labels"] = torch.tensor(e)
        return kod
    return DataLoader(veri, batch_size=a["batch"], shuffle=karistir, collate_fn=birlestir)

def egit_ve_tahmin(X_e, y_e, X_d, a):
    torch.manual_seed(a["seed"])
    m = AutoModelForSequenceClassification.from_pretrained(a["model"], num_labels=2).to("cuda")
    egitim = yukleyici(X_e, y_e, a, karistir=True)
    opt = torch.optim.AdamW(m.parameters(), lr=a["lr"], weight_decay=a["weight_decay"])
    toplam = len(egitim) * a["epoch"]
    plan = get_linear_schedule_with_warmup(opt, int(toplam * a["warmup"]), toplam)

    m.train()
    for ep in range(a["epoch"]):
        for parti in egitim:
            parti = {k: v.to("cuda") for k, v in parti.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                kayip = m(**parti).loss
            kayip.backward()
            opt.step()
            plan.step()
            opt.zero_grad()

    m.eval()
    olasilik = []
    with torch.no_grad():
        for parti in yukleyici(X_d, [0] * len(X_d), a, karistir=False):
            parti = {k: v.to("cuda") for k, v in parti.items() if k != "labels"}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logit = m(**parti).logits
            olasilik.append(torch.softmax(logit.float(), dim=-1)[:, 1].cpu().numpy())

    del m, opt
    torch.cuda.empty_cache()
    return np.concatenate(olasilik)

basla = time.time()
egit, dogr = katlar != 0, katlar == 0
p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit], [X[i] for i in np.where(dogr)[0]], AYAR)
print(f"fold 0 F1: {f1_score(y[dogr], (p > 0.5).astype(int)):.3f}")
print(f"süre: {time.time() - basla:.0f} sn, en yüksek GPU belleği: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import mlflow
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_ikili")

def xlmr_cv(a):
    skorlar = []
    for k in range(5):
        egit, dogr = katlar != k, katlar == k
        p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                           [X[i] for i in np.where(dogr)[0]], a)
        skorlar.append(f1_score(y[dogr], (p > 0.5).astype(int)))
        print(f"fold {k}: {skorlar[-1]:.3f}")
    return np.array(skorlar)

with mlflow.start_run(run_name="xlmr_base"):
    skor = xlmr_cv(AYAR)
    mlflow.log_params(AYAR)
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

In [ ]:
oof = np.zeros(len(y))
for k in range(5):
    egit, dogr = katlar != k, katlar == k
    oof[dogr] = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                               [X[i] for i in np.where(dogr)[0]], AYAR)

train["xlmr_olasilik"] = oof
hatalar = train[(oof > 0.5).astype(int) != y]
print(len(hatalar), "hata,", (hatalar["fold"] == 2).sum(), "tanesi fold 2'de")
hatalar[["id", "fold", "ikili", "xlmr_olasilik", "mesaj"]].sort_values("fold")